# Eval Summary — So sánh adapter vs baseline

Notebook này **không cần GPU**. Chỉ load file JSON eval đã có và vẽ bảng so sánh.

**Pre-flight (Kaggle):**
1. Settings → Accelerator: **None** (không cần GPU)
2. Settings → Add Input: output cũ chứa thư mục `eval/` với các file `*_tier*.json`
3. Run All.

## 1. Locate eval files

In [ ]:
import os, json, glob, shutil
from pathlib import Path
import pandas as pd

# Tìm thư mục eval/ chứa các file JSON tier
EVAL_DIR = None
for root in ["/kaggle/working", "/kaggle/input"]:
    for p in glob.glob(f"{root}/**/eval", recursive=True):
        if Path(p).is_dir() and list(Path(p).glob("*_tier*.json")):
            EVAL_DIR = Path(p)
            break
    if EVAL_DIR:
        break

# Fallback: tìm file JSON tier bất kỳ trong input và copy vào working/eval
if EVAL_DIR is None:
    EVAL_DIR = Path("/kaggle/working/eval")
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    copied = 0
    for f in glob.glob("/kaggle/input/**/*_tier*.json", recursive=True):
        dst = EVAL_DIR / Path(f).name
        if not dst.exists():
            shutil.copy(f, dst)
            copied += 1
    print(f"Copied {copied} files to {EVAL_DIR}")

tier_files = sorted(EVAL_DIR.glob("*_tier*.json"))
print(f"Eval dir : {EVAL_DIR}")
print(f"Files found: {len(tier_files)}")
for f in tier_files:
    print(" ", f.name)

## 2. Detect adapters từ file names

In [ ]:
import re

TIER_SUFFIXES = ["tierA", "tierB1", "tierB2", "tierB3", "tierC"]

# Extract run names từ file names
run_names = set()
for f in tier_files:
    for tier in TIER_SUFFIXES:
        if f.name.endswith(f"_{tier}.json"):
            run_names.add(f.name[: -len(f"_{tier}.json")])

# __base__ luôn đứng đầu
run_names = sorted(run_names, key=lambda x: (x != "__base__", x))
print("Adapters detected:", run_names)

# Check coverage
print()
for run in run_names:
    status = []
    for tier in TIER_SUFFIXES:
        p = EVAL_DIR / f"{run}_{tier}.json"
        status.append(f"{tier}:{'OK' if p.exists() else 'MISSING'}")
    print(f"  {run}: {', '.join(status)}")

## 3. Load kết quả và build bảng so sánh

In [ ]:
def load_metric(run, tier, key):
    p = EVAL_DIR / f"{run}_{tier}.json"
    if not p.exists():
        return None
    data = json.loads(p.read_text())
    return data.get(key)

rows = []
for run in run_names:
    r = {"adapter": run}
    r["tierA_acc"]        = load_metric(run, "tierA",  "accuracy")
    r["tierB1_avg_acc"]   = load_metric(run, "tierB1", "avg_acc")
    r["tierB2_acc"]       = load_metric(run, "tierB2", "accuracy")
    r["tierB3_match"]     = load_metric(run, "tierB3", "loose_match_acc")
    r["tierC_rouge"]      = load_metric(run, "tierC",  "rouge_l_f1")
    r["tierC_bertscore"]  = load_metric(run, "tierC",  "bertscore_f1")  # None nếu không có
    r["tierC_avg_words"]  = load_metric(run, "tierC",  "avg_words")
    rows.append(r)

df = pd.DataFrame(rows)

# Bỏ cột bertscore nếu toàn None
if df["tierC_bertscore"].isna().all():
    df = df.drop(columns=["tierC_bertscore"])

df

## 4. Delta so với __base__

In [ ]:
metric_cols = [c for c in df.columns if c != "adapter"]

base_row = df[df["adapter"] == "__base__"]
if len(base_row) == 0:
    print("Không có __base__ để tính delta.")
else:
    base = base_row.iloc[0]
    df_delta = df.copy()
    for col in metric_cols:
        base_val = base.get(col)
        if base_val is None or pd.isna(base_val):
            continue
        df_delta[f"Δ{col}"] = df[col].apply(
            lambda v: round(v - base_val, 4) if pd.notna(v) else None
        )
    # Chỉ hiện cột delta cho adapter != __base__
    delta_cols = ["adapter"] + [c for c in df_delta.columns if c.startswith("Δ")]
    df_delta[delta_cols][df_delta["adapter"] != "__base__"]

## 5. Selection rule — ACCEPT / REJECT

In [ ]:
def apply_selection(df):
    base_row = df[df["adapter"] == "__base__"]
    if len(base_row) == 0:
        return df.assign(verdict="NO_BASELINE")
    base = base_row.iloc[0]

    verdicts = []
    for _, r in df.iterrows():
        if r["adapter"] == "__base__":
            verdicts.append("BASELINE")
            continue
        reasons = []
        # Gate 1: domain accuracy phải tăng ít nhất 3pt
        if pd.notna(r.get("tierA_acc")) and pd.notna(base.get("tierA_acc")):
            if r["tierA_acc"] < base["tierA_acc"] + 0.03:
                reasons.append(f"tierA +{round((r['tierA_acc']-base['tierA_acc'])*100,1)}pt < 3pt")
        # Gate 2: MMLU không được drop quá 2pt
        if pd.notna(r.get("tierB1_avg_acc")) and pd.notna(base.get("tierB1_avg_acc")):
            if r["tierB1_avg_acc"] < base["tierB1_avg_acc"] - 0.02:
                reasons.append(f"MMLU drop {round((base['tierB1_avg_acc']-r['tierB1_avg_acc'])*100,1)}pt > 2pt")
        # Gate 3 (optional): ROUGE không regression
        if pd.notna(r.get("tierC_rouge")) and pd.notna(base.get("tierC_rouge")):
            if r["tierC_rouge"] < base["tierC_rouge"]:
                reasons.append(f"ROUGE-L regression ({round(base['tierC_rouge'],3)} → {round(r['tierC_rouge'],3)})")
        verdicts.append("ACCEPT" if not reasons else "REJECT: " + " | ".join(reasons))

    return df.assign(verdict=verdicts)

df_v = apply_selection(df)
df_v.to_csv(EVAL_DIR / "summary_with_verdict.csv", index=False)
print("Saved: summary_with_verdict.csv")
df_v

## 6. Bảng so sánh đẹp (styled)

In [ ]:
def highlight_verdict(val):
    if val == "ACCEPT":   return "background-color: #d4edda; color: #155724; font-weight: bold"
    if val == "BASELINE": return "background-color: #e2e3e5; color: #383d41"
    if str(val).startswith("REJECT"): return "background-color: #f8d7da; color: #721c24"
    return ""

def highlight_delta_col(s):
    styles = []
    for v in s:
        if pd.isna(v) or v == "" or v is None:
            styles.append("")
        elif isinstance(v, float):
            styles.append("color: #155724" if v > 0 else ("color: #721c24" if v < 0 else ""))
        else:
            styles.append("")
    return styles

display_cols = ["adapter"] + [c for c in df_v.columns if c not in ["adapter", "verdict"]] + ["verdict"]
df_display = df_v[display_cols]

styled = (
    df_display.style
    .format({c: "{:.4f}" for c in metric_cols if c in df_display.columns}, na_rep="—")
    .applymap(highlight_verdict, subset=["verdict"])
    .set_caption("Adapter Evaluation Summary")
    .set_properties(**{"text-align": "center"})
)

styled

## 7. Detail từng tier (tuỳ chọn xem thêm)

In [ ]:
# Xem chi tiết từng tier của 1 run
def show_detail(run_name):
    print(f"\n=== {run_name} ===")
    for tier in TIER_SUFFIXES:
        p = EVAL_DIR / f"{run_name}_{tier}.json"
        if p.exists():
            data = json.loads(p.read_text())
            print(f"\n  [{tier}]")
            for k, v in data.items():
                if k not in ["run", "tier"]:
                    print(f"    {k}: {v}")
        else:
            print(f"\n  [{tier}] MISSING")

for run in run_names:
    show_detail(run)